<a href="https://colab.research.google.com/github/hUSsAin976-tech/ML-internship-at-FlyRank/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-09 — Validation and Research Claim Audit

Lane: **AI Referral Opportunity** (freestyle). This notebook does two audits: first I read
FlyRank's own research paper (`docs/flyrank-seo-research-march-2026.pdf`) the way the live
session walked it, and write the methodology question I'd ask about two of its findings. Then I
turn the same lens on my own Week-5 model (`w05_model.ipynb`) — an honest-split before/after, a
leakage re-audit of the final feature set, and a rewrite of my own boldest claim in safe language.

Same data contract as ML-04/ML-07/ML-08: `fact_content_daily_performance`, `month=2026-03`
partition only (never `_sample`), joined to `dim_content`. Label stays `has_ai_sessions_month`.

> Working with an AI assistant? Read `skills/README.md`, then load `hunting-leakage-and-validating`
> + `flyrank/flyrank-data` for this task.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and
does the validation design carry the claim? Constructive tone.*

Both findings below come from the ML Appendix of `flyrank-seo-research-march-2026.pdf`
(pp. 26–29) — the exploratory section the paper itself flags as secondary to the direct
aggregate comparisons that lead the report. I picked the appendix on purpose: it's the part of
the paper doing the same kind of work I do in this internship (a fitted model, a holdout split, a
feature-importance readout), so it's the fairest comparison for practicing the next level of
rigor on my own work.

---

### Finding A — "What Predicts Health?" (Random Forest feature importance, p.27)

**The claim:** a holdout-tested Random Forest ranks Average Position (43% importance) and
Impressions (32%) as the top predictors of Health Score, with Scroll Depth (15%) and CTR (8%)
well behind.

**Where does the label come from?** Health Score is defined earlier in the paper (p.5, repeated
in the Methodology section, p.36) as a composite: `Impressions (30 pts) + Position (30 pts) +
CTR (20 pts) + Scroll Depth (20 pts)`. That is the exact feature list the Random Forest is scored
against — Average Position and Impressions alone are worth 60 of the label's 100 possible points.

**My methodology question, respectfully:** the paper already discloses this ("the target itself
is partly constructed from some of these inputs, so importance is descriptive rather than
causal") — that disclosure is exactly right and I don't think this finding is wrong to publish.
My question is whether it goes far enough: the `hunting-leakage-and-validating` skill's
label-derived-features test is to train once WITH the suspect features and once WITHOUT, and
watch whether the score collapses. Here, that test would mean re-running the Random Forest on
Scroll Depth, CTR, Content Age, and Word Count *alone* (holding out Position and Impressions
entirely) and reporting that number next to the headline 43%/32%. If importance collapses toward
the CTR/Scroll levels already shown, the finding is confirmed as "this model is unsurprising
because it's fed its own label"; if some real signal survives, that's a stronger, more
publication-worthy result than the current chart. Either way, the reader can't currently tell
which one is true from the chart alone — the disclosure sentence covers the paper legally, but a
train-without-suspects number would let the finding carry its own weight.

---

### Finding B — "What Predicts Growth?" (Logistic Regression, 71% holdout accuracy, p.28)

**The claim:** a logistic regression separates growing from declining pages at 71% holdout
accuracy, with Content Age as the strongest negative coefficient.

**Where does the label come from?** Growth/decline is `Trend Direction`, computed from the
30-day-vs-previous-30-day impression change (defined p.5): Up is >10% growth, Down is >10%
decline. This is the same label used in Finding #1 (p.6), which reports the underlying counts:
**74,187 rising vs 45,272 falling** pages.

**My methodology question, respectfully, in two parts:**

1. **Base rate isn't printed next to the accuracy.** The `writing-honest-claims` skill's rule is
   "accuracy without base rate next to it" is a banned pattern for exactly this reason. Using the
   paper's own counts from Finding #1, the base rate of the majority class ("growing") in this
   population is about 62% (I check the exact number below) — so 71% accuracy is roughly **9
   points of real skill**, not 71. That's still a real, usable finding, but it reads very
   differently once the base rate sits next to it, and the paper doesn't print that comparison
   anywhere near the 71% figure.
2. **The split isn't described as grouped, and the portfolio spans 57 brands.** The Methodology
   page (p.36) lists this model as an "80/20 split" with no mention of grouping by brand. Pages
   from the same brand plausibly share hidden character — template, existing SEO maturity, how
   aggressively that brand refreshes content — for reasons that have nothing to do with the
   growth signal itself. That is precisely the reasoning this internship's own `flyrank-data` and
   `hunting-leakage-and-validating` skills use to justify grouping ML-08's split by
   `client_hash_id` instead of splitting rows at random. I'd ask: does the 71% holdout accuracy
   still hold on a brand-grouped holdout, where no page from a test-set brand appeared in
   training?

Framed as "how to make it stronger," not a gotcha: both findings are useful and both are already
partially self-aware in their hedging language. My question in each case is the same one the
`writing-honest-claims` skill asks of any bold sentence — would the number survive a
grouped/leakage-controlled re-check — and neither finding currently shows that receipt.

In [1]:
# Light verification of the base-rate arithmetic behind Finding B's methodology question.
# These counts come directly from the paper's own Finding #1 table (p.6) -- no external data
# needed for this section, since I'm auditing the paper's methodology, not re-running its models.

rising = 74187
falling = 45272
base_rate_majority = rising / (rising + falling)

print(f"rising: {rising:,}  falling: {falling:,}  total: {rising + falling:,}")
print(f"base rate of the majority class ('growing'): {base_rate_majority:.1%}")
print(f"reported holdout accuracy of the growth model: 71%")
print(f"apparent skill over always guessing the majority class: "
      f"{(0.71 - base_rate_majority) * 100:.1f} points")

rising: 74,187  falling: 45,272  total: 119,459
base rate of the majority class ('growing'): 62.1%
reported holdout accuracy of the growth model: 71%
apparent skill over always guessing the majority class: 8.9 points


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**What "before" and "after" mean here.** ML-08 (`w05_model.ipynb`) was already built with a
grouped split by `client_hash_id` — I made that call at the time using the same reasoning I just
raised about Finding B: content items from one client share hidden character, and a random
row-level split would let the model partially memorize "which client is this" rather than learn a
transferable pattern. What this notebook adds is the receipt: I re-run the *identical* pipeline
(same query, same features, same Logistic Regression) once with a naive **random split** (the
"before" — what I'd have shipped if I hadn't grouped it) and once with the **grouped split** (the
"after" — what I actually shipped), and report both numbers side by side. Per the
`hunting-leakage-and-validating` skill: "the GAP between them is itself a finding about how much
memorization was happening."

Everything below is the same data contract as ML-08: `month=2026-03`, GA4-available rows only,
demand-worthy slice (`total_gsc_impressions_month >= 100`), same five-feature-plus-freshness
contract, Logistic Regression as the primary (readable) model.

In [2]:
import os, getpass
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass("Paste your Hugging Face READ token (hf_...): ")

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
TABLES = {
    "dim_content":      f"read_parquet('{REL}/dim_content.parquet')",
    "fact_daily_month": f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')",
}

# Identical content_month view to ML-08 -- same contract, same features, so any difference in
# the numbers below comes from the SPLIT, not from a changed feature set.
con.sql(f'''
    CREATE OR REPLACE TEMP VIEW content_month AS
    SELECT
        f.content_hash_id,
        ANY_VALUE(f.client_hash_id)                                                  AS client_hash_id,
        SUM(f.gsc_impressions)                                                       AS total_gsc_impressions_month,
        AVG(CASE WHEN f.gsc_impressions > 0 THEN f.gsc_avg_position END)             AS avg_gsc_position_month,
        COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0 THEN f.report_date END)       AS days_with_impressions_month,
        MAX(CASE WHEN f.ga4_data_available IS TRUE AND f.sessions_ai > 0
                 THEN 1 ELSE 0 END)                                                  AS has_ai_sessions_month
    FROM {TABLES['fact_daily_month']} f
    WHERE f.ga4_data_available IS TRUE
    GROUP BY 1
''')

raw = con.sql(f'''
    SELECT
        cm.content_hash_id,
        cm.client_hash_id,
        cm.total_gsc_impressions_month,
        cm.avg_gsc_position_month,
        cm.days_with_impressions_month,
        cm.has_ai_sessions_month,
        dc.word_count,
        dc.content_type,
        DATE_DIFF('day', dc.content_updated_date, DATE '2026-03-31') AS days_since_update_month
    FROM content_month cm
    JOIN {TABLES['dim_content']} dc USING (content_hash_id)
''').df()
raw["days_since_update_month"] = raw["days_since_update_month"].clip(lower=0)

MIN_DEMAND_IMPRESSIONS = 100
df = raw[raw["total_gsc_impressions_month"] >= MIN_DEMAND_IMPRESSIONS].copy()

print(f"demand-worthy slice (impressions >= {MIN_DEMAND_IMPRESSIONS}): {len(df):,} rows, "
      f"{df['client_hash_id'].nunique()} clients")
print(f"base rate (has_ai_sessions_month == 1): {df['has_ai_sessions_month'].mean():.2%}")

Paste your Hugging Face READ token (hf_...): ··········


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

demand-worthy slice (impressions >= 100): 32,596 rows, 30 clients
base rate (has_ai_sessions_month == 1): 8.61%


In [3]:
# Build BOTH splits on the identical dataframe, then verify each one's honesty receipt:
# for the random split, client overlap is EXPECTED (that's the leak this whole section is about);
# for the grouped split, client overlap must be exactly zero.

from sklearn.model_selection import train_test_split, GroupShuffleSplit

# --- "Before": naive random row-level split ---
train_rand, test_rand = train_test_split(df, test_size=0.25, random_state=42, shuffle=True)
overlap_rand = set(train_rand["client_hash_id"]) & set(test_rand["client_hash_id"])

print("RANDOM split (the 'before')")
print(f"  train: {len(train_rand):,} rows, {train_rand['client_hash_id'].nunique()} clients")
print(f"  test:  {len(test_rand):,} rows, {test_rand['client_hash_id'].nunique()} clients")
print(f"  clients appearing on BOTH sides: {len(overlap_rand)} "
      f"-- this is the leak: most clients show up in both train and test.")

# --- "After": grouped split by client_hash_id (identical to ML-08 Section 2) ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df, groups=df["client_hash_id"]))
train_grp = df.iloc[train_idx].reset_index(drop=True)
test_grp  = df.iloc[test_idx].reset_index(drop=True)
overlap_grp = set(train_grp["client_hash_id"]) & set(test_grp["client_hash_id"])

print("\nGROUPED split (the 'after', same as ML-08)")
print(f"  train: {len(train_grp):,} rows, {train_grp['client_hash_id'].nunique()} clients")
print(f"  test:  {len(test_grp):,} rows, {test_grp['client_hash_id'].nunique()} clients")
print(f"  clients appearing on BOTH sides: {len(overlap_grp)}  (0 means the grouped split held)")
assert len(overlap_grp) == 0, "Group leakage: a client_hash_id appears on both sides."

RANDOM split (the 'before')
  train: 24,447 rows, 28 clients
  test:  8,149 rows, 28 clients
  clients appearing on BOTH sides: 26 -- this is the leak: most clients show up in both train and test.

GROUPED split (the 'after', same as ML-08)
  train: 27,724 rows, 22 clients
  test:  4,872 rows, 8 clients
  clients appearing on BOTH sides: 0  (0 means the grouped split held)


In [4]:
# Train the SAME Logistic Regression pipeline on each split, score with the SAME lift@K
# convention as ML-08, and put both rows in one comparison table.

import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder

NUMERIC_FEATURES = [
    "total_gsc_impressions_month", "avg_gsc_position_month",
    "days_with_impressions_month", "word_count", "days_since_update_month",
]
CATEGORICAL_FEATURES = ["content_type"]
TARGET = "has_ai_sessions_month"

def prep(frame):
    out = frame.copy()
    for c in ["word_count", "avg_gsc_position_month", "days_since_update_month"]:
        out[f"has_{c}"] = out[c].notna().astype(int)
        out[c] = out[c].astype(float).fillna(out[c].median())
    return out

FLAG_FEATURES = [f"has_{c}" for c in ["word_count", "avg_gsc_position_month", "days_since_update_month"]]
ALL_NUMERIC = NUMERIC_FEATURES + FLAG_FEATURES

def make_pipeline():
    preprocess = ColumnTransformer([
        ("num", StandardScaler(), ALL_NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ])
    return Pipeline([
        ("prep", preprocess),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ])

def lift_at_k(scores, y, k):
    order = np.argsort(-np.asarray(scores))
    top_y = np.asarray(y)[order][:k]
    precision = top_y.mean()
    base = np.asarray(y).mean()
    return precision, precision / base

def fit_score(train_df, test_df):
    train_p, test_p = prep(train_df), prep(test_df)
    X_train = train_p[ALL_NUMERIC + CATEGORICAL_FEATURES]
    y_train = train_p[TARGET]
    X_test  = test_p[ALL_NUMERIC + CATEGORICAL_FEATURES]
    y_test  = test_p[TARGET]
    model = make_pipeline().fit(X_train, y_train)
    scores = model.predict_proba(X_test)[:, 1]
    K = max(50, int(round(0.05 * len(test_p))))
    precision, lift = lift_at_k(scores, y_test.values, K)
    return {"K": K, "base_rate": y_test.mean(), "precision@K": precision, "lift@K": lift}

results = {
    "Random split (before)":  fit_score(train_rand, test_rand),
    "Grouped split (after, ML-08)": fit_score(train_grp, test_grp),
}

comparison = pd.DataFrame(results).T
comparison["base_rate"] = comparison["base_rate"].map(lambda v: f"{v:.2%}")
comparison["precision@K"] = comparison["precision@K"].map(lambda v: f"{v:.2%}")
comparison["lift@K"] = comparison["lift@K"].map(lambda v: f"{v:.2f}x")
comparison

,K,base_rate,precision@K,lift@K
Random split (before),407.0,8.01%,44.72%,5.58x
"Grouped split (after, ML-08)",244.0,3.67%,18.44%,5.02x


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

Re-running the `hunting-leakage-and-validating` checklist against the exact feature set trained
above (`total_gsc_impressions_month`, `avg_gsc_position_month`, `days_with_impressions_month`,
`word_count`, `days_since_update_month`, `content_type`, plus the three `has_*` missingness
flags):

- [x] **Timeline drawn:** every feature is summed/averaged from the `month=2026-03` partition
  only; the label (`has_ai_sessions_month`) is read from GA4 sessions in that same month, not a
  future window. This is the same population-and-window design ML-04 validated; I'm not
  re-deriving it here, just confirming the final feature list didn't drift from it.
- [x] **No label-derived or sibling columns:** `sessions_ai_month` (the raw AI-session count
  `has_ai_sessions_month` is thresholded from) was identified as leaky back in ML-04 and has never
  entered the feature set in any of ML-07/ML-08/this notebook. I re-demonstrate the "add it back,
  watch the score jump" test below as this audit's own receipt, rather than just citing the ML-04
  finding by name.
- [x] **No product flags / existing-system scores:** confirmed in ML-07 Section 4 (no
  `priority_score`, `health_score`, or FlyRank workflow flag used anywhere in the score) and
  unchanged in ML-08/this notebook.
- [x] **Population selection checked for outcome-window information:** the demand-worthy filter
  (`total_gsc_impressions_month >= 100`) is computed from the SAME month as the label, not a
  later month — it doesn't smuggle in information from after the label window. Worth naming as a
  disclosed choice per the skill: this filter does mean the validated slice is not identical to
  the zero-AI-session action-queue population ML-07 delivers (see ML-08 Section 1 for why lift@K
  needs positives in the evaluation set).
- [x] **Split grouped by the repeating entity:** confirmed in Section 2 above, with the zero-
  overlap assertion as the receipt.
- [x] **Base rate printed next to every metric:** done in Section 2's comparison table.
- [x] **Top feature importance sanity-checked:** re-run below.
- [x] **Metrics recomputed out-of-fold:** all `lift@K`/`precision@K` numbers above are scored on
  held-out test rows only, never on training rows.
- [ ] **Sealed/holdout claims:** N/A this notebook — no sealed-final-month claim is made here;
  `month=2026-03` is a mid-panel month, not the sealed `_sample` month, per the `flyrank-data`
  skill's iteration rule.

In [5]:
# Deliberate leakage demonstration: add sessions_ai_month (the raw count has_ai_sessions_month
# is thresholded from) back into the feature set and watch lift@K jump toward the ceiling, using
# the GROUPED split from Section 2 so the split itself isn't the variable under test here.

leak_raw = con.sql(f'''
    SELECT
        f.content_hash_id,
        SUM(CASE WHEN f.ga4_data_available IS TRUE THEN f.sessions_ai ELSE 0 END) AS sessions_ai_month
    FROM {TABLES['fact_daily_month']} f
    GROUP BY 1
''').df()

df_leak = df.merge(leak_raw, on="content_hash_id", how="left")
df_leak["sessions_ai_month"] = df_leak["sessions_ai_month"].fillna(0)

train_leak = df_leak.iloc[train_idx].reset_index(drop=True)
test_leak  = df_leak.iloc[test_idx].reset_index(drop=True)

LEAK_NUMERIC = ALL_NUMERIC + ["sessions_ai_month"]

def fit_score_with_leak(train_df, test_df):
    train_p, test_p = prep(train_df), prep(test_df)
    preprocess = ColumnTransformer([
        ("num", StandardScaler(), LEAK_NUMERIC),
        ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_FEATURES),
    ])
    model = Pipeline([
        ("prep", preprocess),
        ("clf", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
    ]).fit(train_p[LEAK_NUMERIC + CATEGORICAL_FEATURES], train_p[TARGET])
    scores = model.predict_proba(test_p[LEAK_NUMERIC + CATEGORICAL_FEATURES])[:, 1]
    K = max(50, int(round(0.05 * len(test_p))))
    precision, lift = lift_at_k(scores, test_p[TARGET].values, K)
    return {"K": K, "base_rate": test_p[TARGET].mean(), "precision@K": precision, "lift@K": lift}

leak_result   = fit_score_with_leak(train_leak, test_leak)
honest_result = results["Grouped split (after, ML-08)"]

leak_compare = pd.DataFrame({
    "Honest (no sessions_ai_month)": honest_result,
    "WITH sessions_ai_month (deliberate leak)": leak_result,
}).T
leak_compare["base_rate"] = leak_compare["base_rate"].map(lambda v: f"{v:.2%}")
leak_compare["precision@K"] = leak_compare["precision@K"].map(lambda v: f"{v:.2%}")
leak_compare["lift@K"] = leak_compare["lift@K"].map(lambda v: f"{v:.2f}x")
leak_compare

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,K,base_rate,precision@K,lift@K
Honest (no sessions_ai_month),244.0,3.67%,18.44%,5.02x
WITH sessions_ai_month (deliberate leak),244.0,3.67%,73.36%,19.97x


## 4. Claim rewrite

*Take your own boldest sentence from ML-07 or ML-08 and rewrite it using safe language:
observed, measured, directional, decision-support.*

**Original (ML-07, Verdict A, Section 1):**

> "A same-shape check on the starter dataset's equivalent columns... showed a clean, monotonic
> climb: 1.6%... → 14.3%... — a roughly 9x gap between the bottom and top bucket, against a 6.4%
> base rate."

This sentence is already fairly careful — it names the base rate and the sample sizes, which
`writing-honest-claims` specifically asks for. The place it goes a little further than the
evidence, on a close read, is the word **"clean"**: it's describing a real, large, monotonic
pattern, but "clean" reads as more certain/complete than a single cross-sectional bucket check on
one month's data actually supports — it doesn't (and can't, from this design) rule out that the
9x gap is partly a proxy for something else volume correlates with (e.g., brand size, content
age), the same confounding the FlyRank paper itself names on its own Methodology page for a
similar variable.

**Rewrite, using the claim ladder:**

> "In this month's data, higher search-impression volume is **associated with** a substantially
> higher observed rate of AI-referred sessions: roughly 1.6% in the lowest bucket versus 14.3% in
> the highest (a **directional**, ~9x gap), against a 6.4% base rate across n=30,000 starter-set
> rows. This is a **decision-support** signal for prioritizing which pages to review first, not a
> causal claim that volume drives AI referral — the two could share a common cause (e.g., overall
> content or brand strength) that this single-snapshot check can't separate out."

**What changed and why:**
- "showed a clean... climb" → "is associated with... a directional... gap" (claim ladder: pattern
  → association, not certainty)
- Added the explicit non-causal caveat and named the specific alternative explanation
  (confounding), instead of leaving it implicit
- Kept the concrete numbers and base rate exactly as they were — the fix is in the verb and the
  hedge, not in hiding the effect size behind vaguer language; per the skill, "effect size over
  drama" cuts both ways: don't inflate the words, but don't strip the numbers either.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.